In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

### Read Bronze table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

In [0]:
df.display()

# Silver Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Customer ID Cleanup

In [0]:
df = df.withColumn("cid", f.regexp_replace(col("cid"), "-", ""))

### Country Normalization

In [0]:
df = df.withColumn(
    "cntry",
    f.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)

### Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity checks of dataframe

In [0]:
df.limit(10).display()

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

### Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location